# lets_route — `route_f` Playground

This notebook runs on the **`route_f`** kernel (Python 3.12, uv-managed).
The same environment is used in the terminal via:

```bash
source .route_f/bin/activate
```

> If the kernel isn't listed in VS Code yet, finish the setup first:
> `./setup_route_f.sh` (or verify with `jupyter kernelspec list`).

In [ ]:
import sys, platform
print(f"Python : {sys.version.split()[0]} (executable: {sys.executable})")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"torch : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")
    try:
        torch.zeros(1, device=torch.cuda.is_available() and 'cuda' or 'cpu')
        print("tensor allocation ok")
    except Exception as e:
        print("tensor allocation failed:", e)
except ImportError:
    print("torch not installed yet — run ./setup_route_f.sh")

In [ ]:
# Quick sanity check of the training stack
import importlib
mods = ["transformers", "datasets", "accelerate", "peft", "pandas", "numpy", "pyarrow"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:16s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{m:16s} NOT INSTALLED")

## Dataset: xRouteBench (`ulab-ai/xRouteBench`)

LLM-routing benchmark — each row is a `(query, candidate model)` pair with the
model's response, token usage, latency, and a `performance` score. A router
learns which model to pick per query, trading off performance vs price.

- **Configs (scenarios):** `llmrouter_generic`, `memory_locomo`, `memory_longmemeval`,
  `timeseries`, `video`, `multimodal_geometry3k`, `multimodal_mathvista`, `personalized`.
- **Raw queries:** `<scenario>_queries` (train/valid/test) — no model executions.
- **Pricing pool:** `llm_candidates` (18 models with prices).
- **No bulk download:** the cells below inspect everything via the HF API and
  stream only a few rows (parquet pulled lazily). Set upstream=False`` only when
  you actually want to download the full dataset for training.

In [ ]:
import sys
sys.path.insert(0, ".")
from data.load_xroutebench import list_configs, schema, peek

print("Available configs (from HF datasets-server API, no download):")
for c in list_configs():
    print("  -", c)

cols, samples = schema("llmrouter_generic", "train")
print("\nSchema for llmrouter_generic/train:")
print(cols)

print("\nFirst 2 rows via STREAMING (only the first parquet chunk is read):")
for row in peek("llmrouter_generic", "train", n=2):
    print("  -", {k: row[k] for k in ("task_name", "model_name", "metric")})